In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18
import torch.nn.functional as F

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data Augmentation and Normalization
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

batch_size = 64

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False)

net = resnet18(weights=None)
net.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
net.fc = nn.Linear(net.fc.in_features, 10)
net = net.to(device)

# 3. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# 4. Training Loop
num_epochs = 20
for epoch in range(num_epochs):
    net.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / len(trainloader):.4f}")

print("Finished Training")

# 5. Accuracy on Test Set
net.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


Epoch 1/20, Loss: 1.3350
Epoch 2/20, Loss: 0.8613
Epoch 3/20, Loss: 0.6754
Epoch 4/20, Loss: 0.5737
Epoch 5/20, Loss: 0.5024
Epoch 6/20, Loss: 0.4435
Epoch 7/20, Loss: 0.4044
Epoch 8/20, Loss: 0.3672
Epoch 9/20, Loss: 0.3311
Epoch 10/20, Loss: 0.3037
Epoch 11/20, Loss: 0.2212
Epoch 12/20, Loss: 0.1978
Epoch 13/20, Loss: 0.1793
Epoch 14/20, Loss: 0.1684
Epoch 15/20, Loss: 0.1569
Epoch 16/20, Loss: 0.1455
Epoch 17/20, Loss: 0.1337
Epoch 18/20, Loss: 0.1284
Epoch 19/20, Loss: 0.1168
Epoch 20/20, Loss: 0.1081
Finished Training
Test Accuracy: 91.12%
